In [1]:
import numpy as np
import pandas as pd
import Soil_Carbon_Model as md
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit 
import os
import openpyxl
from pandas import *
%matplotlib inline

In [2]:
import pylab
params = {'backend': 'ps',
          'axes.labelsize': 20,
          'legend.fontsize': 20,
          'legend.handlelength': 2.0,
          'legend.borderaxespad': 0,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'font.family': 'serif',
          'font.size': 20,
          'font.weight': 'normal',
          #'font.serif': ['Palatino'],
          'ps.usedistiller': 'xpdf',
          'text.usetex': False,
         'agg.path.chunksize' : 1000}
pylab.rcParams.update(params)

In [3]:
dt = 1 / 24
nyears = 2000
ndays = nyears * 365
nsim = ndays * 24
nnpp = ndays * 24

#Rainfall from Poisson
rain = md.poisson_process(n = int(nsim), alpha = 11.88/1000, lbd = (0.22 * dt))

tsim = np.linspace(1, nnpp, int(nnpp)) # Hour

days = np.arange(365)
annual_avg_npp = 560

# Half sine wave shape over 365 days
shape = np.sin((np.pi*days)/365)                       # sinB(x-h) + K; B = pi/365, h = 0, K = 0

# Normalize so that the sum equals the annual NPP
daily_npp = annual_avg_npp * (shape / np.sum(shape))
hourly_npp = np.repeat(daily_npp, 24)

npp = np.tile(hourly_npp,nyears)

In [4]:
print("Maximum Value is:", max(npp))
print("Length is:", len(npp))
print("Average value is:", np.mean(npp))

Maximum Value is: 2.4099814458967175
Length is: 17520000
Average value is: 1.534246575342465


In [5]:
total_rainfall = rain.sum()
print(f"Total rainfall over 450 years = {total_rainfall:.3f} meters")

RPY = (total_rainfall / nyears)*1000         # average rainfall per year
print(f"Average rainfall per year = {RPY:.3f} mm/year")

Total rainfall over 450 years = 1906.744 meters
Average rainfall per year = 953.372 mm/year


In [6]:
PET = 884.04
AI = PET/RPY
print(AI)

"""
def hill_constrained(R, KA, n):
    R_max = 4.18

    numerator = R**n
    denominator = KA**n + R**n
    norm_factor = R_max**n / (KA**n + R_max**n)
    TI = (numerator / denominator) / norm_factor

    # Clip to ensure TI does not exceed 1
    return (np.minimum(TI, 1.0))

TI = hill_constrained(AI, 3.45, 2.28)
print(TI)
"""

# Final best-fit parameters from your latest run
KA_final = 9.0132
n_final = 1.7444
R_max = 3.8  # Max AI used for normalization

def compute_TI(AI):
    AI = np.asarray(AI)  # Accept scalars or arrays
    numerator = AI**n_final
    denominator = KA_final**n_final + AI**n_final
    norm_factor = R_max**n_final / (KA_final**n_final + R_max**n_final)
    TI = (numerator / denominator) / norm_factor
    return np.minimum(TI, 1.0)

TI = compute_TI(AI)
print("TI =", TI)

0.9272770552836099
TI = 0.10238394477998605


In [7]:
#Hydraulic parameterization for dynamic porosity -------------------------------
alpha1 = 580  #hydraulic conductivity shape parameter
beta1 = 6     #hydraulic conductivity shape parameter
alpha2 = 580  #porosity shape parameter
beta2 = 6     #porosity shape parameter
kstex = 0.1   #m/d,
kstot = 1     #m/d
phitex = 0.50  
phitot = 0.60  

In [8]:
# Simulation for 2150 years at dynamic porosity
sim = md.run_julia(rain, npp, kstex, kstot, phitex, phitot, alpha1, beta1, alpha2, beta2, TI)
sim_year = sim.pivot_table(values=['s', 'phi', 'ksstr', 'P', 'DOC', 'A', 'M', 'B',
                                   'Fa', 'Fpa','Flb','Fl', 'Flm', 'Fpl', 'Fbm','Fld', 'Fal',
                                   'lk','npp_', 'L','Fi', 'Flpom', 'decomp'],
                           index='Years')

print(sim_year)

100%|██████████| 17520000/17520000 [48:49<00:00, 5981.43it/s]  


                 A           B         DOC        Fa       Fal       Fbm  \
Years                                                                      
1       915.262637   73.158478   21.250345  0.128525  0.003520  0.265752   
2      1031.852647   22.022892   18.550277  0.142262  0.002450  0.080000   
3      1150.768168    6.651157   18.266106  0.159480  0.001245  0.024161   
4      1263.485310    2.016409   19.040424  0.174802  0.000491  0.007325   
5      1365.085871    0.615224   19.994413  0.189987  0.000174  0.002235   
...            ...         ...         ...       ...       ...       ...   
1996   1790.169477  161.877438  244.592722  0.249295  0.008069  0.587950   
1997   1790.207320  158.133530  243.590081  0.248074  0.008004  0.574349   
1998   1790.249390  174.890833  234.185394  0.241482  0.007892  0.635211   
1999   1790.355142  185.700338  225.206277  0.243705  0.008020  0.674469   
2000   1790.429780  190.987250  220.180902  0.247587  0.008175  0.693681   

           

In [11]:
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Puget & Lal\PGL.csv", index= True, header = True)

sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Results\PGL.csv", index= True, header = True)